In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║            BanglaBLIP — Colab Backend (FastAPI + ngrok)                     ║
# ║  Run each cell in order. GPU runtime required.                              ║
# ╚══════════════════════════════════════════════════════════════════════════════╝


# ┌─────────────────────────────────────────────────────────────────────────────┐
# │ CELL 1 — Install dependencies                                               │
# └─────────────────────────────────────────────────────────────────────────────┘

import subprocess, sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "git+https://github.com/csebuetnlp/normalizer"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-Uq",
    "bitsandbytes", "transformers", "peft", "safetensors",
    "fastapi", "uvicorn[standard]", "python-multipart",
    "pyngrok", "nest-asyncio", "huggingface_hub"])


# ┌─────────────────────────────────────────────────────────────────────────────┐
# │ CELL 2 — Configuration (edit these before running)                          │
# └─────────────────────────────────────────────────────────────────────────────┘

# ── Your HuggingFace token (from huggingface.co/settings/tokens) ──────────────
HF_TOKEN       = "YOUR_HF_TOKEN"

# ── Your model repo on HF Hub ─────────────────────────────────────────────────
HF_MODEL_REPO  = "Suprio85/BanglaBLIP-30k"

# ── ngrok auth token (free at ngrok.com) ─────────────────────────────────────
NGROK_TOKEN    = "3BSxAifEzd3rIXy1ml0arW5Tz02_2uMvAWFwZu4mUa9e5Aino"

# ── Where the model will be downloaded on Colab ───────────────────────────────
CHECKPOINT_DIR = "/content/model"

# ── "lora" | "full_finetune" ──────────────────────────────────────────────────
MODEL_MODE     = "lora"

from huggingface_hub import snapshot_download

print(f"Downloading model from {HF_MODEL_REPO} ...")
snapshot_download(
    repo_id=HF_MODEL_REPO,
    repo_type="model",
    local_dir=CHECKPOINT_DIR,
)
print(f"Model downloaded to {CHECKPOINT_DIR}")


# ┌─────────────────────────────────────────────────────────────────────────────┐
# │ CELL 4 — Imports, Config class, BanglaBLIP model definition                 │
# └─────────────────────────────────────────────────────────────────────────────┘

import warnings, os, json, random, io, urllib.request
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings('ignore', message='.*unexpected keys.*')
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
import torch.nn as nn
from PIL import Image
from transformers import (
    Blip2Processor, Blip2PreTrainedModel,
    Blip2VisionModel, Blip2QFormerModel,
    AutoTokenizer, AutoModelForSeq2SeqLM, AutoConfig,
    BitsAndBytesConfig
)
from peft import PeftModel


# ── Config ────────────────────────────────────────────────────────────────────
class Config:
    blip_checkpoint    = "Salesforce/blip2-flan-t5-xl"
    bangla_model_id    = "csebuetnlp/banglat5"

    max_len            = 128
    image_size         = 224
    max_gen_length     = 64
    num_beams          = 5
    length_penalty     = 1.5

    device = "cuda" if torch.cuda.is_available() else "cpu"

config = Config()
print(f"Using device: {config.device}")


# ── Model Definition (exact copy from training) ───────────────────────────────
class BanglaBLIP(Blip2PreTrainedModel):
    def __init__(self, blip_pretrained="Salesforce/blip2-flan-t5-xl",
                 bangla_lm_pretrained="csebuetnlp/banglat5",
                 load_8bit=True, freeze_vit=True, freeze_qformer=True,
                 freeze_lm=True, freeze_projection=False):

        from transformers import Blip2ForConditionalGeneration
        blip2_model = Blip2ForConditionalGeneration.from_pretrained(
            blip_pretrained, torch_dtype=torch.float16)
        cfg = blip2_model.config
        bangla_config = AutoConfig.from_pretrained(bangla_lm_pretrained)
        cfg.text_config = bangla_config
        super().__init__(cfg)

        self.vision_model  = blip2_model.vision_model
        self.qformer       = blip2_model.qformer
        self.query_tokens  = blip2_model.query_tokens

        self.language_projection = nn.Sequential(
            nn.Linear(self.qformer.config.hidden_size, bangla_config.d_model),
            nn.GELU(),
            nn.Linear(bangla_config.d_model, bangla_config.d_model))
        self.language_projection_ln = nn.LayerNorm(bangla_config.d_model)

        del blip2_model.language_model; del blip2_model
        torch.cuda.empty_cache()

        rank = int(os.environ.get("LOCAL_RANK", 0))
        self.llm_cast_dtype = torch.bfloat16

        if MODEL_MODE == "lora" and load_8bit:
            bnb_config = BitsAndBytesConfig(load_in_8bit=True)
            self.language_model = AutoModelForSeq2SeqLM.from_pretrained(
                bangla_lm_pretrained, quantization_config=bnb_config,
                device_map={"": rank})
        else:
            self.language_model = AutoModelForSeq2SeqLM.from_pretrained(
                bangla_lm_pretrained, torch_dtype=torch.float16,
                device_map={"": rank})

        # Freeze everything for inference
        for p in self.parameters(): p.requires_grad = False

    def get_input_embeddings(self):
        return self.language_model.get_input_embeddings()

    def set_input_embeddings(self, value):
        self.language_model.set_input_embeddings(value)

    @torch.no_grad()
    def generate(self, pixel_values, input_ids=None, attention_mask=None,
                 **generate_kwargs):
        num_images = 1
        orig_batch_size = pixel_values.shape[0]

        if len(pixel_values.shape) == 5:
            orig_batch_size = pixel_values.shape[0]
            num_images      = pixel_values.shape[1]
            pixel_values    = pixel_values.view(
                pixel_values.shape[0] * pixel_values.shape[1],
                *pixel_values.shape[2:])

        image_embeds = self.vision_model(
            pixel_values, return_dict=True).last_hidden_state
        image_attention_mask = torch.ones(
            image_embeds.size()[:-1], dtype=torch.long,
            device=image_embeds.device)

        query_tokens = self.query_tokens.expand(image_embeds.shape[0], -1, -1)
        query_outputs = self.qformer(
            query_embeds=query_tokens,
            encoder_hidden_states=image_embeds,
            encoder_attention_mask=image_attention_mask,
            return_dict=True)
        query_output = query_outputs.last_hidden_state

        if num_images > 1:
            query_output = query_output.view(
                orig_batch_size, -1, query_output.shape[2])

        language_model_inputs = self.language_projection(query_output)
        language_model_inputs = self.language_projection_ln(language_model_inputs)
        language_attention_mask = torch.ones(
            language_model_inputs.size()[:-1], dtype=torch.long,
            device=language_model_inputs.device)

        if input_ids is None:
            input_ids = (
                torch.LongTensor([[self.config.text_config.bos_token_id]])
                .repeat(orig_batch_size, 1)
                .to(image_embeds.device))
        if attention_mask is None:
            attention_mask = torch.ones_like(input_ids)

        attention_mask = torch.cat(
            [language_attention_mask, attention_mask], dim=1)

        with torch.cuda.amp.autocast(dtype=self.llm_cast_dtype):
            lm_embedding  = self.language_model.get_input_embeddings()
            inputs_embeds = lm_embedding(input_ids)
            inputs_embeds = torch.cat(
                [language_model_inputs,
                 inputs_embeds.to(language_model_inputs.device)], dim=1)
            outputs = self.language_model.generate(
                inputs_embeds=inputs_embeds,
                attention_mask=attention_mask,
                **generate_kwargs)
        return outputs


# ┌─────────────────────────────────────────────────────────────────────────────┐
# │ CELL 5 — load_model() — exact copy from your notebook                       │
# └─────────────────────────────────────────────────────────────────────────────┘

def load_model():
    print(f"Loading model from: {CHECKPOINT_DIR}")
    print(f"Mode: {MODEL_MODE}")

    processor = Blip2Processor.from_pretrained(config.blip_checkpoint)
    print("Processor loaded")

    tokenizer = AutoTokenizer.from_pretrained(config.bangla_model_id)
    print("Tokenizer loaded")

    use_8bit = (MODEL_MODE == "lora")
    model = BanglaBLIP(
        blip_pretrained=config.blip_checkpoint,
        bangla_lm_pretrained=config.bangla_model_id,
        load_8bit=use_8bit)

    # Load saved weights
    checkpoint_file = os.path.join(CHECKPOINT_DIR, "model.safetensors")
    if not os.path.exists(checkpoint_file):
        checkpoint_file = os.path.join(CHECKPOINT_DIR, "pytorch_model.bin")

    if os.path.exists(checkpoint_file):
        print(f"Loading checkpoint: {checkpoint_file}")
        if checkpoint_file.endswith(".safetensors"):
            from safetensors.torch import load_file
            state_dict = load_file(checkpoint_file)
        else:
            state_dict = torch.load(checkpoint_file, map_location="cpu")

        missing, unexpected = model.load_state_dict(state_dict, strict=False)
        print(f"Loaded checkpoint — missing: {len(missing)}, unexpected: {len(unexpected)}")
    else:
        print(f"WARNING: No checkpoint file found in {CHECKPOINT_DIR}")

    # Load LoRA adapter if applicable
    if MODEL_MODE == "lora":
        lora_dir = os.path.join(CHECKPOINT_DIR, "lora_adapter")
        if not os.path.exists(lora_dir):
            lora_dir = CHECKPOINT_DIR

        adapter_config = os.path.join(lora_dir, "adapter_config.json")
        if os.path.exists(adapter_config):
            print(f"Loading LoRA adapter from: {lora_dir}")
            model.language_model = PeftModel.from_pretrained(
                model.language_model, lora_dir)
            print("LoRA adapter loaded")
        else:
            print("No LoRA adapter_config.json found, skipping LoRA loading")

    model.eval()
    model.to(config.device)
    print("Model ready for inference\n")
    return model, processor, tokenizer


# ┌─────────────────────────────────────────────────────────────────────────────┐
# │ CELL 6 — run_inference() — based on your infer_single_image()               │
# │          (do_sample removed; beam search only as you confirmed)             │
# └─────────────────────────────────────────────────────────────────────────────┘

def run_inference(image: Image.Image, model, processor, tokenizer) -> str:
    model.eval()
    device = config.device

    pixel_values = processor(
        images=image, return_tensors="pt").pixel_values.to(device)

    instruction  = "বাংলায় ক্যাপশন:"
    text_encoding = tokenizer(
        instruction,
        padding="max_length",
        max_length=config.max_len,
        truncation=True,
        return_tensors="pt")
    input_ids      = text_encoding["input_ids"].to(device)
    attention_mask = text_encoding["attention_mask"].to(device)

    with torch.no_grad():
        output_ids = model.generate(
            pixel_values=pixel_values,
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_length=config.max_gen_length,
            num_beams=5,
            length_penalty=1.7,
            no_repeat_ngram_size=2,
            repetition_penalty=1.2,
            early_stopping=True,
            # do_sample removed — conflicts with beam search
        )

    caption = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return caption


# ┌─────────────────────────────────────────────────────────────────────────────┐
# │ CELL 7 — Actually load the model (run once, takes a few minutes)            │
# └─────────────────────────────────────────────────────────────────────────────┘

model, processor, tokenizer = load_model()


# ┌─────────────────────────────────────────────────────────────────────────────┐
# │ CELL 8 — FastAPI app + ngrok tunnel + uvicorn                               │
# └─────────────────────────────────────────────────────────────────────────────┘

import nest_asyncio, uvicorn, threading
from pyngrok import ngrok
from fastapi import FastAPI, File, UploadFile, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

nest_asyncio.apply()   # required to run uvicorn inside Colab's event loop

app = FastAPI(title="BanglaBLIP Caption API")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],    # tighten this if you go to production
    allow_methods=["*"],
    allow_headers=["*"],
)

# ── Endpoint 1: image file upload ─────────────────────────────────────────────
@app.post("/caption/upload")
async def caption_from_upload(file: UploadFile = File(...)):
    if not file.content_type.startswith("image/"):
        raise HTTPException(status_code=400, detail="File must be an image.")
    contents = await file.read()
    try:
        image = Image.open(io.BytesIO(contents)).convert("RGB")
    except Exception as e:
        raise HTTPException(status_code=400, detail=f"Could not open image: {e}")
    caption = run_inference(image, model, processor, tokenizer)
    return {"caption": caption}

# ── Endpoint 2: image URL ──────────────────────────────────────────────────────
class URLRequest(BaseModel):
    url: str

@app.post("/caption/url")
async def caption_from_url(body: URLRequest):
    try:
        req      = urllib.request.Request(
            body.url, headers={"User-Agent": "Mozilla/5.0"})
        response = urllib.request.urlopen(req, timeout=10)
        image    = Image.open(io.BytesIO(response.read())).convert("RGB")
    except Exception as e:
        raise HTTPException(status_code=400,
                            detail=f"Could not load image from URL: {e}")
    caption = run_inference(image, model, processor, tokenizer)
    return {"caption": caption}

# ── Health check ───────────────────────────────────────────────────────────────
@app.get("/health")
def health():
    return {"status": "ok"}

# ── Start ngrok tunnel ─────────────────────────────────────────────────────────
ngrok.set_auth_token(NGROK_TOKEN)
public_url = ngrok.connect(8000).public_url
print(f"\n{'='*55}")
print(f"  ✅  BanglaBLIP backend is live!")
print(f"  🌐  Base URL    : {public_url}")
print(f"  📤  Upload      : {public_url}/caption/upload")
print(f"  🔗  URL         : {public_url}/caption/url")
print(f"  💓  Health      : {public_url}/health")
print(f"  📖  Docs        : {public_url}/docs")
print(f"{'='*55}\n")

# ── Start server (blocks — keep this cell running) ────────────────────────────
import threading

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

thread = threading.Thread(target=run_server, daemon=True)
thread.start()

print("Server is running in the background.")
print("The cell will not block — you can run other cells now.")



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Model downloaded to /content/model
Using device: cuda
Loading model from: /content/model
Mode: lora


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Loading checkpoint: /content/model/model.safetensors
Loaded checkpoint — missing: 758, unexpected: 0
Loading LoRA adapter from: /content/model/lora_adapter
LoRA adapter loaded
Model ready for inference


  ✅  BanglaBLIP backend is live!
  🌐  Base URL    : https://pokily-endamoebic-kaitlyn.ngrok-free.dev
  📤  Upload      : https://pokily-endamoebic-kaitlyn.ngrok-free.dev/caption/upload
  🔗  URL         : https://pokily-endamoebic-kaitlyn.ngrok-free.dev/caption/url
  💓  Health      : https://pokily-endamoebic-kaitlyn.ngrok-free.dev/health
  📖  Docs        : https://pokily-endamoebic-kaitlyn.ngrok-free.dev/docs

Server is running in the background.
The cell will not block — you can run other cells now.


In [ ]:
import requests
r = requests.post(f"{public_url}/caption/url",
                  json={"url": "https://cdn.pixabay.com/photo/2016/04/05/11/04/india-1309206_1280.jpg"})
print(r.json())

INFO:     Started server process [2684]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     34.142.182.225:0 - "POST /caption/url HTTP/1.1" 200 OK
{'caption': 'একটি রৌদ্রোজ্জ্বল দিনে একটি নৌকায় চারজন লোক বসে আছে'}
